### img classification 과 object detection 을 결합하여 분석 진행한다.

In [1]:
import pandas as pd

features = pd.read_csv("image_features.csv")  # 객체탐지 feature
class_probs = pd.read_csv("classification_features_iteration2.csv")  # 방금 만든 확률

# 병합
df = features.merge(class_probs, on="image", how="inner")

# accident_label 다시 생성
df["accident_label"] = df["image"].apply(
    lambda x: 0 if x.endswith("북쪽.png") else 1
)

print(df.columns)
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'image_features.csv'

In [3]:
# 회귀분석 모델링
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# 입력 변수
X = df[[
    "p_wide",
    "p_barrier_yes",
    "parked_density",
    "sidewalk_ratio",
    "road_width_relative"
]]

y = df["accident_label"]

# 스케일링 포함 파이프라인 (권장)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])

model.fit(X, y)

# 계수 확인
coef = model.named_steps["logit"].coef_[0]
intercept = model.named_steps["logit"].intercept_[0]

results = pd.DataFrame({
    "variable": X.columns,
    "coefficient": coef,
    "odds_ratio": np.exp(coef)
})

print("Intercept:", intercept)
print(results)

Intercept: -0.30248912314887616
              variable  coefficient  odds_ratio
0               p_wide     1.236999    3.445260
1        p_barrier_yes    -0.804222    0.447436
2       parked_density     0.109352    1.115555
3       sidewalk_ratio    -0.137334    0.871679
4  road_width_relative     0.246574    1.279634


In [5]:
#회귀분석 2 
import statsmodels.api as sm
import numpy as np

X = df[[
    "p_wide",
    "p_barrier_yes",
    "parked_density",
    "sidewalk_ratio",
    "road_width_relative"
]]

X = sm.add_constant(X)
y = df["accident_label"]

model = sm.Logit(y, X).fit()
print(model.summary())

# 오즈비 계산
odds_ratios = np.exp(model.params)
print("\nOdds Ratios:")
print(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.611239
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:         accident_label   No. Observations:                 3971
Model:                          Logit   Df Residuals:                     3965
Method:                           MLE   Df Model:                            5
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                  0.1070
Time:                        10:54:18   Log-Likelihood:                -2427.2
converged:                       True   LL-Null:                       -2717.9
Covariance Type:            nonrobust   LLR p-value:                2.079e-123
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -1.6764      0.110    -15.286      0.000      -1.891      -1.461
p_

In [4]:
#모델 성능 확인
from sklearn.metrics import roc_auc_score, accuracy_score

y_pred_prob = model.predict_proba(X)[:, 1]
y_pred = model.predict(X)

print("Accuracy:", accuracy_score(y, y_pred))
print("ROC-AUC:", roc_auc_score(y, y_pred_prob))

Accuracy: 0.6655754218081088
ROC-AUC: 0.7142746141542681


#### 다변량 로지스틱 회귀 결과 요약
모델:
```
accident_label ~ 
p_wide +
p_barrier_yes +
parked_density +
sidewalk_ratio +
road_width_relative
```
Pseudo R² = **0.107**
→ 구조 변수 5개로 약 10.7% 설명력 확보
(2변수 모델 9.6% → 유의하게 상승)

1️⃣ p_wide\
coef = 3.504\
OR ≈ **33.25**\
p < 0.001 \
넓은 도로 특성은 사고다발지 확률을 강하게 증가시킨다.

2️⃣ p_barrier_yes\
coef = -2.662\
OR ≈ **0.07**\
p < 0.001\
물리적 분리장치는 사고 확률을 크게 감소시킨다. -> 강한 보호 효과.

3️⃣ road_width_relative \
coef = 1.694 \
OR ≈ **5.44**\
p < 0.001\
실제 도로 상대 폭이 넓을수록 사고 확률 증가. \
p_wide 를 보강하며 구조적 폭 효과가 확실히 존재함을 보인다.

4️⃣ parked_density\
coef = 0.120\
OR ≈ **1.13**\
p = 0.004\
주정차 차량 밀도 증가 → 사고 확률 소폭 증가.
효과는 작지만 통계적으로 유의.

5️⃣ sidewalk_ratio\
coef = -1.163\
OR ≈ **0.31**\
p = 0.002\
보행공간 비율이 높을수록 사고 확률 감소.
 즉, 보행 공간 확보가 보호 효과를 가진다.


#### 📌 구조적 결론
1. 넓은 도로는 사고 위험 증가 요인.
2. 물리적 분리장치는 강력한 보호 요인.
3. 보행 공간 확보는 위험 감소 요인.
4. 주정차 밀도는 보조 위험 요인.
5. 도로 실제 폭도 독립적으로 위험 요인.\
\
🧠 중요한 점 : p_wide + road_width_relative 둘 다 유의. \
        → 단순 모델 편향이 아니라 실제 구조적 도로 특성이 사고 분포와 연결됨.

##### 📊 설명력
Pseudo R² = 0.107
 * 도시 구조 변수만으로는 중간 수준 설명력
 * 사고는 구조 + 교통량 + 인구 + 시간요인 복합 결과
지금 단계에서는 충분히 의미 있는 수준.
#### 📎 보고서용 요약 문장
> 다변량 로지스틱 회귀 분석 결과, 도로 폭 관련 변수(p_wide, road_width_relative)는 사고다발지 형성과 강하게 양(+)의 관계를 보였으며, \
물리적 분리장치(p_barrier_yes) 및 보행공간 비율(sidewalk_ratio)은 통계적으로 유의한 보호 효과를 나타냈다. \
이는 스쿨존 사고다발지의 형성이 단순한 위치 특성이 아닌 구조적 도로 환경과 밀접하게 연관되어 있음을 시사한다.